In [ ]:
!pip install chromadb sentence-transformers groq pandas --q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 60.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 78.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 65.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currentl

In [ ]:
import pandas as pd
import chromadb

from sentence_transformers import SentenceTransformer
from groq import Groq
import os

print("All libraries imported successfully!")
print("Ready to build RAG system")

All libraries imported successfully!
Ready to build RAG system


In [ ]:

GROQ_API_KEY=""
os.environ["GROQ_API_KEY"]=GROQ_API_KEY
groq_client=Groq(api_key=GROQ_API_KEY)

print("Groq API client initialized successfully!")
print("Ready to build RAG system")

Groq API client initialized successfully!
Ready to build RAG system


In [ ]:
df=pd.read_csv('college_notes.csv')

print("Shape of dataset:",df.shape)
print("\nColumn names:",df.columns.tolist())
print("\nFirst 3 rows:")
print(df.head(3))

FileNotFoundError: [Errno 2] No such file or directory: 'college_notes.csv'

In [ ]:
print("Subjects in the dataset:")
print(df['subject'].value_counts())

print("\nSample of topics:")
print(df[['note_id','subject','topic']].to_string(index=False))
print("\nLength of content (number of characters) for each note:")
df['content_length']=df['content'].apply(len)
print(df[['topic','content_length']].to_string(index=False))

In [ ]:
documents=df['content'].tolist()
ids=[f"note_{row['note_id']}"for row in df.to_dict('records')]
metadatas=[
    {"subject": row['subject'],"topic": row['topic']}
    for row in df.to_dict('records')
]
print(f"total chunks prepared: {len(documents)}")
print(f"First document ID : {ids[0]}")
print(f"First document metadata : {metadatas[0]}")
print(f"First 100 chars of doc: {documents[0][:100]}...")

In [ ]:
embedding_model=SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
test_embedding=embedding_model.encode("This is a test sentence")

print(f"Test embedding shape: {test_embedding.shape}")
print(f"First 5 values of a test embedding: {test_embedding[:5]}")

In [ ]:
chroma_client=chromadb.Client()
collection=chroma_client.get_or_create_collection(name="college_notes_rag")

print("Chroma client created.")
print(f"Collection name: college_notes_rag")
print(f"Documents in collection so far: {collection.count()}")

In [ ]:
embeddings=embedding_model.encode(documents,show_progress_bar=True)
print(f"\nEmbedding matrix shape: {embeddings.shape}")
embeddings_list=embeddings.tolist()

collection.add(
    documents=documents,
    metadatas=metadatas,
    ids=ids,
    embeddings=embeddings_list
)
print()

In [ ]:
embeddings=embedding_model.encode(documents,show_progress_bar=True)
print(f"\nEmbedding matrix shape: {embeddings.shape}")
embeddings_list=embeddings.tolist()

collection.add(
    documents=documents,
    metadatas=metadatas,
    ids=ids,
    embeddings=embeddings_list
)
print(f"\nDocumments successfully addedd to collection")
print(f"Documents in collection now: {collection.count()}")

In [ ]:
def retrieve_relevent_chunks(question,top_k=3):
    query_embedding=embedding_model.encode(question).tolist()
    results=collection.query(
        query_embeddings=query_embedding,
        n_results=top_k,

    )
    return results

In [ ]:
test_question="what is ETL and how does it work in data engineering "
print(f"test question:{test_question}")
print("=" *60)
results=retrieve_relevent_chunks(test_question,top_k=3)

print("\top 3 retruved chunks:")
print("="*60)
for i ,(doc,dist,meta) in enumerate (zip(
    results['document'][0],
    results['distance'][0],
    results['metadatas'][0],
)):
    print(f"\nResult {i+1}:")
    print(f"sub:{meta['subject']}")
    print(f"topic:{meta['topic']}")
    print(f"distance:{dist:.4f}")
    print(f"content :{doc[:120]}...")

    for i,(doc,dist,meta) in enumerate(zip(
        results['documents'][0],

        results['metadatas'][0],
    )):


In [ ]:
def build_context_from_results(results):

    context_parts=[]
    for i,(doc,dist,meta) in enumerate(zip(
            results['documents'][0],
            results['metadatas'][0],
        )):
        # Assuming 'subject' and 'topic' are strings and should be concatenated
        chunk_test=f"[source {i+1}:{meta['subject']}-{meta['topic']}]\n{doc}"
        context_parts.append(chunk_test)
    context_str="\n\n---\n\n".join(context_parts)
    return context_str

# Example usage (moved outside the function and corrected)
# This part was present in the original cell but was unreachable or incorrectly placed.
# It's kept here as a demonstration of how to use the function.
      results_example = {'documents': [['doc1', 'doc2']], 'metadatas': [{'subject': 'Sub1', 'topic': 'Top1'}, {'subject': 'Sub2', 'topic': 'Top2'}]}
      context = build_context_from_results(results_example)
      print("built context:")
      print("="*60)
      print(context)
      print("="*60)

In [ ]:
def generate_rag_answer(question, conrext)

system_prompt = """You are a helpful academic assistant for engineering students.


    You will be given context retrieved from a college knowledge base, and a student's question.


    RULES:
    1. Answer ONLY using the information provided in the context below.
    2. If the answer is not found in the context, say exactly:
       "I don't have enough information in my knowledge base to answer this question."
    3. Do not use your general training knowledge.
    4. Keep answers clear, accurate, and beginner-friendly.
    5. Mention which source the information came from when possible."""

user_prompt = f"""Context from Knowledge Base:

{context}

---

Student's Question: {question}

Please answer the question based only on the context provided above."""

response = groq_client.chat.completions.create(
     model="llama-3.1-8b-instant",
     messages=[
         {
             "role": "system",
             "content": system_prompt
         },
         {
             "role": "user",
             "content": user_prompt
         }
     ],
     temperature=0.1,

     max_tokens=500
)

answer = response.choices[0].message.content

return answer

print("RAG generation function defined.")

In [ ]:
def ask_college_assistant(question, top_k=3, verbose=True):

  if verbose:
    print(f"Question: {question}")
    print("=" * 60)
    print("Step 1: Retrieve relevant documents...")

  results = retrieve_relevent_chunks(question, top_k=top_k)

  if verbose:
    print(f"Retrived {top_k}chunks from Knowledge Base:")
    for i, meta in enumerate(results['metadatas'][0]):
      print(f" {i+1}. {meta['subject']} - {meta['topic']}")
    print("\nStep 2: building context string...")

  context = build_context_from_results(results)

  if verbose:
    print(f"Context built ({len(context)} character)")
    print("\nStep 3: sending to LLM for answer generation...")

  answer = generate_rag_answer(question, context)

  if verbose:
    print("\n" + "=" * 60)
    print("ANSWER:")
    print("=" * 60)
    print(answer)
    print("=" * 60)

  return answer

print("complete RAG pipeline function ready.")
print("Function ask_college_assistant(question, top_k=3)")

In [ ]:
question_1="waht is ETL and what are it three main stages?"

answer_1=ask_college_assistant(question_1,top_k=3,verbose=True)

Question: waht is ETL and what are it three main stages?
Step 1: Retrieve relevant documents...


NameError: name 'retrieve_relevent_chunks' is not defined